# IEEE-118: independent evaluation scenarios for PF and masked-voltage SE

This notebook loads the fine-tuned 250,000-scenario IEEE-118 **PowerFlow** checkpoint and its saved normalizer from [gridfm/genco-pf-transfer](https://huggingface.co/gridfm/genco-pf-transfer). It evaluates every scenario in the separate IEEE-118 evaluation dataset (9,952 scenarios according to the model card) twice. The GENCO reproduction guide says this dataset is distinct from the fine-tuning run's own train/validation/test split, so these are held-out scenarios for this checkpoint. The IEEE-118 grid topology itself was seen during fine-tuning:

- **PF:** Graphkit's PowerFlow task transform applies the checkpoint's standard power-flow masks.
- **SE:** the same PF checkpoint estimates voltages after 10% of bus `Vm` and 10% of bus `Va` measurements are hidden independently. Other bus, generator, and branch measurements remain available, as in the previous masked-voltage experiment. This is a measurement-masking experiment, not a separately trained state-estimation model or Graphkit's noisy StateEstimation task.

Both runs save one row per bus with scenario identity, ground truth, raw model predictions, and voltage-mask flags. `Vm` is in p.u., `Va` in degrees, and `Pg`/`Qg` in MW/Mvar. Predictions for known (unmasked) values are saved as emitted by the model; they are not replaced with ground truth. The dataset contains its original contingency cases; no outages are generated here.

Run from the `gridfm-graphkit` repository root in an environment with its dependencies, `huggingface_hub`, `pandas`, `pyarrow`, and `tqdm`. The first run downloads model artifacts and may build Graphkit's processed dataset cache. The two Parquet files are written under `data/ieee118_predictions/`.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import torch
import yaml
from huggingface_hub import hf_hub_download, snapshot_download
from torch_geometric.data import Batch
from tqdm.auto import tqdm

from gridfm_graphkit.cli import _normalize_loaded_state_dict_keys
from gridfm_graphkit.datasets.globals import (
    PQ_H,
    PV_H,
    REF_H,
    VA_H,
    VM_H,
    PG_OUT,
    QG_OUT,
    VA_OUT,
    VM_OUT,
    QG_H,
)
from gridfm_graphkit.datasets.powergrid_hetero_dataset import HeteroGridDatasetDisk
from gridfm_graphkit.datasets.transforms import (
    RemoveInactiveBranches,
    RemoveInactiveGenerators,
)
from gridfm_graphkit.io.param_handler import (
    NestedNamespace,
    get_task_transforms,
    load_model,
    load_normalizer,
)

MODEL_REPO_ID = "gridfm/genco-pf-transfer"
MODEL_REVISION = "main"
RUN_DIR = Path("finetune/case118_250000")
NETWORK = "case118_ieee"
DATA_ROOT = Path("data/hf-genco-pf-transfer/data")
OUTPUT_DIR = Path("results/original_predictions")
BATCH_SIZE = 16
SEED = 42
SE_MASK_FRACTION = 0.10
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Inference device: {DEVICE}")

Inference device: cuda


## 1. Download and load the matched artifacts and full evaluation dataset

The checkpoint, YAML, and normalizer are from the same fine-tuning run. The downloaded `gridfm/genco-pf-transfer/data/case118_ieee/raw/` is a separate evaluation dataset, distinct from the fine-tuning dataset `gridfm/pf_small_case118_ieee` and its internal test split. This notebook iterates over all downloaded evaluation scenarios. The team's paper evaluation uses an evaluation YAML with `test_ratio: 0.99`; using every scenario here is therefore not the exact paper evaluation partition, although all come from the separate evaluation dataset. See the [GENCO transfer reproduction instructions](https://github.com/gridfm/gridfm-paper-genco/blob/main/repro_instructions/5.5.3_PF_transfer.md).


In [2]:
artifact_paths = {
    key: hf_hub_download(
        MODEL_REPO_ID,
        f"{RUN_DIR}/{filename}",
        revision=MODEL_REVISION,
    )
    for key, filename in {
        "checkpoint": "best_model_state_dict.pt",
        "config": "case118_250000.yaml",
        "normalizer": "normalizer_stats.pt",
    }.items()
}
snapshot_download(
    repo_id=MODEL_REPO_ID,
    revision=MODEL_REVISION,
    allow_patterns=[f"data/{NETWORK}/raw/**"],
    local_dir=DATA_ROOT.parent,
)
raw_dir = DATA_ROOT / NETWORK / "raw"
required_tables = ["bus_data.parquet", "gen_data.parquet", "branch_data.parquet"]

with open(artifact_paths["config"], encoding="utf-8") as handle:
    args = NestedNamespace(**yaml.safe_load(handle))

saved_stats = torch.load(
    artifact_paths["normalizer"],
    map_location="cpu",
    weights_only=True,
)
normalizer = load_normalizer(args)
normalizer.fit_from_dict(saved_stats[NETWORK])
dataset = HeteroGridDatasetDisk(
    root=str(DATA_ROOT / NETWORK),
    data_normalizer=normalizer,
    transform=None,
    stream_partitions=getattr(args.data, "stream_partitions", "auto"),
)

model = load_model(args)
state = torch.load(
    artifact_paths["checkpoint"],
    map_location="cpu",
    weights_only=True,
)
state = _normalize_loaded_state_dict_keys(state)
model.load_state_dict(
    {key.removeprefix("model."): value for key, value in state.items()},
    strict=True,
)
model.to(DEVICE).eval()
print(f"Loaded {len(dataset):,} evaluation scenarios from {raw_dir}")

finetune/case118_250000/best_model_state(…): reconstructing file:   0%|          |  0.00B / 80.5MB            

finetune/case118_250000/best_model_state(…): downloading bytes:           |  0.00B            

case118_250000.yaml:   0%|          | 0.00/803 [00:00<?, ?B/s]

finetune/case118_250000/normalizer_stats(…): reconstructing file:   0%|          |  0.00B / 2.16kB            

finetune/case118_250000/normalizer_stats(…): downloading bytes:           |  0.00B            

...

Fetching 295 files:   0%|          | 0/295 [00:00<?, ?it/s]

Processing...


Detected 50 Hive partitions — using streaming mode.
Streaming 50 partitions...





















































Processing partitions: 100%|██████████| 50/50 [02:45<00:00,  3.31s/it]
Done!


Loaded 9,952 evaluation scenarios from data/hf-genco-pf-transfer/data/case118_ieee/raw


## 2. Define PF and SE inputs

PF uses Graphkit's registered PowerFlow transform, including removal of inactive equipment and the task's standard masks. For SE, inactive equipment is removed first; a fixed seed derived from each dataset index chooses the hidden voltage measurements. The target `y` tensors are preserved before input masking. The per-scenario seed makes SE masks independent of batch size and execution order.


In [3]:
pf_transform = get_task_transforms(args)
remove_branches = RemoveInactiveBranches()
remove_generators = RemoveInactiveGenerators()
mask_value = float(args.data.mask_value)


def prepare_graph(index, task):
    graph = dataset[index]
    if task == "pf":
        return pf_transform(graph)
    if task != "se":
        raise ValueError(f"Unknown task: {task}")

    graph = remove_generators(remove_branches(graph))
    bus_x = graph["bus"].x
    n_bus = bus_x.size(0)
    n_mask = max(1, round(SE_MASK_FRACTION * n_bus))
    rng = np.random.default_rng(np.random.SeedSequence([SEED, int(index)]))
    bus_mask = torch.zeros_like(bus_x, dtype=torch.bool)
    bus_mask[torch.as_tensor(rng.choice(n_bus, n_mask, replace=False)), VM_H] = True
    bus_mask[torch.as_tensor(rng.choice(n_bus, n_mask, replace=False)), VA_H] = True
    graph.mask_dict = {
        "bus": bus_mask,
        "PQ": bus_x[:, PQ_H] == 1,
        "PV": bus_x[:, PV_H] == 1,
        "REF": bus_x[:, REF_H] == 1,
        "gen": torch.zeros_like(graph["gen"].x, dtype=torch.bool),
        "branch": torch.zeros_like(
            graph["bus", "connects", "bus"].edge_attr,
            dtype=torch.bool,
        ),
    }
    bus_x[bus_mask] = mask_value
    return graph

In [4]:
prepare_graph(0, "pf")

HeteroData(
  scenario_id=[1],
  baseMVA=137.93067932128906,
  is_normalized=True,
  mask_dict={
    bus=[118, 15],
    gen=[53, 6],
    branch=[372, 10],
    PQ=[118],
    PV=[118],
    REF=[118],
  },
  bus={
    x=[118, 15],
    static=[118, 5],
    y=[118, 5],
  },
  gen={
    x=[53, 6],
    y=[53, 1],
  },
  (bus, connects, bus)={
    edge_index=[2, 372],
    edge_attr=[372, 10],
    static=[372, 3],
    y=[372, 2],
  },
  (gen, connected_to, bus)={ edge_index=[2, 53] },
  (bus, connected_to, gen)={ edge_index=[2, 53] }
)

In [5]:
prepare_graph(0, "se")

HeteroData(
  scenario_id=[1],
  baseMVA=137.93067932128906,
  is_normalized=True,
  mask_dict={
    bus=[118, 15],
    PQ=[118],
    PV=[118],
    REF=[118],
    gen=[53, 6],
    branch=[372, 10],
  },
  bus={
    x=[118, 15],
    static=[118, 5],
    y=[118, 5],
  },
  gen={
    x=[53, 6],
    y=[53, 1],
  },
  (bus, connects, bus)={
    edge_index=[2, 372],
    edge_attr=[372, 10],
    static=[372, 3],
    y=[372, 2],
  },
  (gen, connected_to, bus)={ edge_index=[2, 53] },
  (bus, connected_to, gen)={ edge_index=[2, 53] }
)

## 3. Predict every scenario and write ground truth with predictions

Rows are streamed to Parquet one batch at a time to avoid retaining the full test output in memory. The `dataset_index`, `scenario_id`, and zero-based `bus_index` columns identify every row. `vm_masked` and `va_masked` tell which voltage values were unavailable to the model. The checkpoint's bus head also emits `Pg` and `Qg`; their ground truths are aggregate generator `Pg` at each bus and the bus `Qg` label.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def predict_all(task):
    output_path = OUTPUT_DIR / f"case118_ieee_{task}_ground_truth_predictions.parquet"
    temp_path = output_path.with_suffix(".tmp.parquet")
    writer = None
    rows_written = 0
    seen_scenarios = set()
    with torch.inference_mode():
        for start in tqdm(
            range(0, len(dataset), BATCH_SIZE),
            desc=f"IEEE-118 {task.upper()}",
        ):
            indices = list(range(start, min(start + BATCH_SIZE, len(dataset))))
            graphs = [prepare_graph(index, task) for index in indices]
            batch = Batch.from_data_list(graphs).to(DEVICE)
            output = model(batch)
            pred = output["bus"]
            if pred.size(1) <= max(VM_OUT, VA_OUT, PG_OUT, QG_OUT):
                raise ValueError("Model bus output does not contain Vm, Va, Pg and Qg")

            # Graphkit keeps Va in radians during inverse_transform.
            normalizer.inverse_output(output, batch)
            normalizer.inverse_transform(batch)
            bus_y = batch["bus"].y
            gen_y = batch["gen"].y
            gen_to_bus = batch["gen", "connected_to", "bus"].edge_index[1]
            pg_target = torch.zeros(
                bus_y.size(0),
                device=DEVICE,
                dtype=gen_y.dtype,
            )
            pg_target.index_add_(0, gen_to_bus, gen_y[:, 0])

            ptr = batch["bus"].ptr.cpu().tolist()
            frames = []
            for j, index in enumerate(indices):
                a, b = ptr[j], ptr[j + 1]
                scenario_id = int(graphs[j].scenario_id.item())
                if scenario_id in seen_scenarios:
                    raise ValueError(f"Duplicate scenario_id: {scenario_id}")
                seen_scenarios.add(scenario_id)
                mask = batch.mask_dict["bus"][a:b]
                frames.append(
                    pd.DataFrame(
                        {
                            "dataset_index": np.full(b - a, index, dtype=np.int32),
                            "scenario_id": np.full(b - a, scenario_id, dtype=np.int64),
                            "bus_index": np.arange(b - a, dtype=np.int32),
                            "pq_bus": batch.mask_dict["PQ"][a:b].cpu().numpy(),
                            "pv_bus": batch.mask_dict["PV"][a:b].cpu().numpy(),
                            "ref_bus": batch.mask_dict["REF"][a:b].cpu().numpy(),
                            "vm_masked": mask[:, VM_H].cpu().numpy(),
                            "va_masked": mask[:, VA_H].cpu().numpy(),
                            "vm_gt_pu": bus_y[a:b, VM_H].cpu().numpy(),
                            "vm_pred_pu": pred[a:b, VM_OUT].cpu().numpy(),
                            "va_gt_deg": torch.rad2deg(bus_y[a:b, VA_H]).cpu().numpy(),
                            "va_pred_deg": torch.rad2deg(pred[a:b, VA_OUT])
                            .cpu()
                            .numpy(),
                            "pg_gt_mw": pg_target[a:b].cpu().numpy(),
                            "pg_pred_mw": pred[a:b, PG_OUT].cpu().numpy(),
                            "qg_gt_mvar": bus_y[a:b, QG_H].cpu().numpy(),
                            "qg_pred_mvar": pred[a:b, QG_OUT].cpu().numpy(),
                        }
                    )
                )
            table = pa.Table.from_pandas(
                pd.concat(frames, ignore_index=True),
                preserve_index=False,
            )
            if writer is None:
                writer = pq.ParquetWriter(temp_path, table.schema, compression="zstd")
            writer.write_table(table)
            rows_written += table.num_rows
    if writer is None:
        raise RuntimeError(f"No rows were written for task {task!r}")
    writer.close()
    temp_path.replace(output_path)
    print(f"Saved {rows_written:,} rows to {output_path}")
    return output_path


pf_path = predict_all("pf")
se_path = predict_all("se")

IEEE-118 PF:   0%|          | 0/622 [00:00<?, ?it/s]

## 4. Check the saved files

The two files must have one row for every bus in every evaluation scenario and identical `(dataset_index, scenario_id, bus_index)` keys. These checks read only identifiers, keeping memory use modest.


In [ ]:
key_columns = ["dataset_index", "scenario_id", "bus_index"]
pf_keys = pd.read_parquet(pf_path, columns=key_columns)
se_keys = pd.read_parquet(se_path, columns=key_columns)

print("PF:", pf_path)
print("SE:", se_path)

FileNotFoundError: [Errno 2] No such file or directory: 'results/original_predictions/case118_ieee_pf_ground_truth_predictions.parquet'

In [ ]:
pd.read_parquet(pf_path, columns=key_columns)